# Graph-completion benchmark report: reproducible supplement

Recomputes every table in `docs/reference/benchmark-report-graph-completion.md` from the committed run data under `bench/results/graph-completion-confirmatory/` and `bench/results/graph-completion-probe/`, and regenerates every figure. Tables need no network, no API key, and no dependencies beyond the standard library; figures additionally need matplotlib. The table logic lives in `graph_tables.py` beside this notebook, and `make bench-report-check` runs it as a build gate.

In [ ]:
import graph_tables as gt

# Where the data comes from:
print(gt.CONFIRMATORY)
print(gt.PROBE)

runs = gt.load_confirmatory()
probe_runs = gt.load_probe()
agg = gt.aggregate(runs)
print(gt.manifest_pins(runs))

## T1-T3 — the confirmatory matrix, provenance, and the elicited claim

The seven-condition matrix (design doc, "Arms and matrix (frozen for #1251)"): episode means for grounded coverage, the discontinuity reading, cost, and traversal; then the share of fetches by where each reference was first seen; then the elicited completeness claim per condition.

In [ ]:
gt.t1_matrix(agg)
gt.t2_provenance(agg)
claims = gt.t3_claims(agg)

## T4-T5 — the instrument kill, surfaced as the recorded outcome

The pre-registered instrument kill fired: stripped-arm episodes grounded the certified-unreachable discontinuity constraints at both certified scales. This is the study's recorded outcome, present in the archives by design — the archived analyzer (`graph-confirmatory-analyze.py`, kept beside the run directories) exits non-zero on these archives for exactly this reason, and `check_pins` below asserts that it still does. T5 prints the kill-condition readings, which per the frozen design are informational, never confirmatory findings.

In [ ]:
leaks = gt.t4_kill(runs, agg)

## T6 — the pilot (graph-completion probe, #1241)

The probe that validated the instruments, aggregated the way its own archived analyzer aggregates. Its role in the report is pilot evidence: the no-search floors (stripped 0.00 at every reading budget, graph 0.96/0.42) and the budget-bound cells, never the headline claims.

In [ ]:
probe = gt.t6_probe(probe_runs)

## T7 — the published-headline pin

Every headline number the published report prints, recomputed and compared, including the kill's presence and the archived analyzer's non-zero exit. A nonzero return means the archives no longer reproduce the report.

In [ ]:
failed = gt.check_pins(agg, leaks, claims, probe)
assert failed == 0, f"{failed} published-headline pin(s) failed" 

## Figures

Regenerates the report's three figures into `figures/` and `docs/reference/benchmark-figures/graph-completion/`. Requires matplotlib; everything is computed from the archives via `graph_tables.py`, nothing is hand-entered.

In [ ]:
import figures
figures.main()